## ANALISIS DE LAS BASES DE DATOS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración para mostrar todas las columnas en los outputs
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

def analizar_dataset_completo(rutas_archivos):
    """
    Analiza múltiples archivos CSV y genera un reporte consolidado.
    Args:
        rutas_archivos (dict): Diccionario {'Nombre descriptivo': 'ruta/al/archivo.csv'}
    """
    for nombre, ruta in rutas_archivos.items():
        print(f"\n{'='*80}")
        print(f"  ANÁLISIS DE BASE DE DATOS: {nombre.upper()}")
        print(f"{'='*80}\n")

        try:
            # Carga del dataframe
            df = pd.read_csv(ruta)

            # 1. Vista General
            print(f"--- [1] VISTA GENERAL ---")
            print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
            print(f"Memoria usada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")

            # 2. Análisis de Nulos y Tipos
            print(f"--- [2] ESTRUCTURA DE DATOS Y CALIDAD (NULOS) ---")
            info_df = pd.DataFrame({
                'Tipo de Dato': df.dtypes,
                'Nulos (Cant)': df.isnull().sum(),
                'Nulos (%)': (df.isnull().sum() / len(df)) * 100,
                'Valores Únicos': df.nunique()
            })
            # Ordenar por porcentaje de nulos descendente para identificar problemas rápido
            print(info_df.sort_values('Nulos (%)', ascending=False))
            print("\n")

            # 3. Análisis de Columnas Clave (Diagnósticos y Metadatos)
            print(f"--- [3] ANÁLISIS DE CONTENIDO (COLUMNAS CLAVE) ---")
            # Detectamos automáticamente columnas que parecen diagnósticos o categorías importantes
            cols_interes = [col for col in df.columns if df[col].dtype == 'object' or df[col].nunique() < 50]

            for col in cols_interes:
                num_unicos = df[col].nunique()
                # Si tiene pocos valores únicos (ej. < 50), es una categoría analizable
                if num_unicos < 50 and num_unicos > 0:
                    print(f"\n> Distribución de valores en '{col}':")
                    # Mostramos conteo absoluto y relativo
                    val_counts = df[col].value_counts()
                    val_norm = df[col].value_counts(normalize=True) * 100
                    resumen = pd.concat([val_counts, val_norm], axis=1, keys=['Cantidad', 'Porcentaje (%)'])
                    print(resumen)

            # 4. Análisis Específico de Diagnósticos (si existen columnas específicas)
            diag_cols = [c for c in df.columns if 'diagnosis' in c.lower() or 'iddx' in c.lower() or 'malignant' in c.lower()]
            if diag_cols:
                 print(f"\n--- [4] FOCO EN DIAGNÓSTICOS ENCONTRADOS ---")
                 for col in diag_cols:
                     if df[col].nunique() >= 50: # Si son muchos diagnósticos, mostrar solo el top 20
                         print(f"\n> Top 20 valores en columna diagnóstica compleja '{col}':")
                         print(df[col].value_counts().head(20))

        except Exception as e:
            print(f"ERROR al procesar el archivo {ruta}: {e}")

# --- EJEMPLO DE USO ---
# Reemplaza las rutas con las ubicaciones reales de tus archivos en tu PC
mis_archivos = {
    'Slice 3D Metadata': 'C:\\Users\\pbarc\\OneDrive - Universidad Loyola Andalucía\\TFG\\metadata.csv',
    'ISIC Ground Truth': 'C:\\Users\\pbarc\\OneDrive - Universidad Loyola Andalucía\\TFG\\ISIC_2024_Training_GroundTruth.csv',
    'ISIC Supplement': 'C:\\Users\\pbarc\\OneDrive - Universidad Loyola Andalucía\\TFG\\ISIC_2024_Training_Supplement.csv'
}

analizar_dataset_completo(mis_archivos)

# Ejecutar la función
# analizar_dataset_completo(mis_archivos)


  ANÁLISIS DE BASE DE DATOS: SLICE 3D METADATA

ERROR al procesar el archivo C:\Users\pbarc\OneDrive - Universidad Loyola Andalucía\TFG\metadata.csv: [Errno 2] No such file or directory: 'C:\\Users\\pbarc\\OneDrive - Universidad Loyola Andalucía\\TFG\\metadata.csv'

  ANÁLISIS DE BASE DE DATOS: ISIC GROUND TRUTH

--- [1] VISTA GENERAL ---
Dimensiones: 401059 filas x 2 columnas
Memoria usada: 26.39 MB

--- [2] ESTRUCTURA DE DATOS Y CALIDAD (NULOS) ---
          Tipo de Dato  Nulos (Cant)  Nulos (%)  Valores Únicos
isic_id         object             0        0.0          401059
malignant      float64             0        0.0               2


--- [3] ANÁLISIS DE CONTENIDO (COLUMNAS CLAVE) ---

> Distribución de valores en 'malignant':
           Cantidad  Porcentaje (%)
malignant                          
0.0          400666       99.902009
1.0             393        0.097991

--- [4] FOCO EN DIAGNÓSTICOS ENCONTRADOS ---

  ANÁLISIS DE BASE DE DATOS: ISIC SUPPLEMENT



C:\Users\pbarc\AppData\Local\Temp\ipykernel_10060\2067681891.py:23: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta)


--- [1] VISTA GENERAL ---
Dimensiones: 401059 filas x 13 columnas
Memoria usada: 205.11 MB

--- [2] ESTRUCTURA DE DATOS Y CALIDAD (NULOS) ---
                             Tipo de Dato  Nulos (Cant)  Nulos (%)  \
iddx_5                             object        401058  99.999751   
mel_mitotic_index                  object        401006  99.986785   
mel_thick_mm                      float64        400996  99.984292   
iddx_4                             object        400508  99.862614   
iddx_3                             object        399994  99.734453   
iddx_2                             object        399991  99.733705   
lesion_id                          object        379001  94.500061   
attribution                        object             0   0.000000   
isic_id                            object             0   0.000000   
iddx_full                          object             0   0.000000   
copyright_license                  object             0   0.000000   
iddx_1            

## Estudiamos metadata_malignant patient_id y lesion_id

In [1]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
# Apuntamos al archivo de la BD Maligna
FILE_MALIGNANT = 'metadata_malignant.csv' 
COLS_TO_ANALYZE = ['patient_id', 'lesion_id']

print(f">>> 📈 AUDITORÍA DE NULOS (NaN) EN '{FILE_MALIGNANT}'...")

# ==============================================================================
# 2. CARGA Y ANÁLISIS
# ==============================================================================
try:
    # Cargar solo las columnas 'patient_id' y 'lesion_id' para ahorrar memoria
    df = pd.read_csv(FILE_MALIGNANT, usecols=COLS_TO_ANALYZE, low_memory=False)
    total_rows = len(df)
    print(f"✅ Archivo cargado. Total de {total_rows:,} filas (imágenes) a analizar.")

except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{FILE_MALIGNANT}'.")
    print("Asegúrate de que el script esté en la misma carpeta que el archivo.")
    exit()
except ValueError:
    print(f"❌ ERROR: El archivo '{FILE_MALIGNANT}' no contiene las columnas {COLS_TO_ANALYZE}.")
    exit()
except Exception as e:
    print(f"❌ ERROR inesperado: {e}")
    exit()

# --- 1. Conteo de NaNs en patient_id ---
patient_id_nans = df['patient_id'].isna().sum()
patient_id_nans_pct = (patient_id_nans / total_rows) * 100

# --- 2. Conteo de NaNs en lesion_id ---
lesion_id_nans = df['lesion_id'].isna().sum()
lesion_id_nans_pct = (lesion_id_nans / total_rows) * 100

# --- 3. Conteo de NaNs simultáneos ---
# Busca filas donde 'patient_id' ES Nulo Y 'lesion_id' ES Nulo
both_nans = df[df['patient_id'].isna() & df['lesion_id'].isna()].shape[0]
both_nans_pct = (both_nans / total_rows) * 100

# ==============================================================================
# 3. REPORTE DE RESULTADOS
# ==============================================================================
print("\n--- [ RESULTADOS DE LA AUDITORÍA DE NULOS ] ---")

print(f"\n1. 'patient_id':")
print(f"   - Filas con 'patient_id' Nulo (NaN): {patient_id_nans:,} (de {total_rows:,})")
print(f"   - Porcentaje de 'patient_id' Nulos: {patient_id_nans_pct:.2f}%")

print(f"\n2. 'lesion_id':")
print(f"   - Filas con 'lesion_id' Nulo (NaN): {lesion_id_nans:,} (de {total_rows:,})")
print(f"   - Porcentaje de 'lesion_id' Nulos: {lesion_id_nans_pct:.2f}%")

print(f"\n3. NaNs SIMULTÁNEOS:")
print(f"   - Filas donde AMBOS ('patient_id' Y 'lesion_id') son Nulos: {both_nans:,}")
print(f"   - Porcentaje de NaNs Simultáneos: {both_nans_pct:.2f}%")

print("\n>>> 🎉 AUDITORÍA TERMINADA.")

>>> 📈 AUDITORÍA DE NULOS (NaN) EN 'metadata_malignant.csv'...
✅ Archivo cargado. Total de 27,069 filas (imágenes) a analizar.

--- [ RESULTADOS DE LA AUDITORÍA DE NULOS ] ---

1. 'patient_id':
   - Filas con 'patient_id' Nulo (NaN): 20,450 (de 27,069)
   - Porcentaje de 'patient_id' Nulos: 75.55%

2. 'lesion_id':
   - Filas con 'lesion_id' Nulo (NaN): 4,024 (de 27,069)
   - Porcentaje de 'lesion_id' Nulos: 14.87%

3. NaNs SIMULTÁNEOS:
   - Filas donde AMBOS ('patient_id' Y 'lesion_id') son Nulos: 1,210
   - Porcentaje de NaNs Simultáneos: 4.47%

>>> 🎉 AUDITORÍA TERMINADA.


## Fusion inteligente de lesion_id en NAN patient_id

In [1]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
# Este es el archivo que subiste (BD Maligna)
FILE_IN = 'metadata_malignant.csv' 
# Este es el nuevo archivo que crearemos
FILE_OUT = 'metadata_malignant_grouped.csv' 

COLS_TO_LOAD = ['isic_id', 'patient_id', 'lesion_id']

print(f">>> 🚀 INICIANDO AGRUPACIÓN INTELIGENTE (v3.0) en '{FILE_IN}'...")

# ==============================================================================
# 2. CARGA Y LÓGICA DE AGRUPACIÓN JERÁRQUICA
# ==============================================================================
try:
    # Cargamos el dataframe completo, no solo las columnas de ID, 
    # para no perder los datos de diagnóstico, edad, etc.
    df = pd.read_csv(FILE_IN, low_memory=False)
    print(f"✅ Archivo '{FILE_IN}' cargado. {len(df):,} filas totales.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{FILE_IN}'.")
    exit()
except Exception as e:
    print(f"❌ ERROR inesperado: {e}")
    exit()

# Verificar que las columnas clave existen
required_cols = ['isic_id', 'patient_id', 'lesion_id']
if not all(col in df.columns for col in required_cols):
    print(f"❌ ERROR: El archivo no contiene las columnas necesarias: {required_cols}")
    exit()

print("   -> Aplicando lógica de agrupación jerárquica...")

# --- Esta es la "Agrupación Inteligente" ---
# 1. Prioridad 1: Usar 'patient_id' si existe.
# 2. Prioridad 2: Si no, usar 'lesion_id' si existe.
# 3. Prioridad 3: Si ambos faltan, usar 'isic_id' (cada imagen es su propio grupo).
df['master_group_id'] = np.where(
    df['patient_id'].notna(),  # Condición 1
    'p_' + df['patient_id'].astype(str), # Si es Verdadero (usar patient_id)
    np.where(
        df['lesion_id'].notna(), # Condición 2
        'l_' + df['lesion_id'].astype(str), # Si es Verdadero (usar lesion_id)
        'i_' + df['isic_id'].astype(str)    # Si es Falso (usar isic_id)
    )
)

# ==============================================================================
# 3. VERIFICACIÓN Y GUARDADO
# ==============================================================================

# --- Verificación de la Lógica ---
print("\n--- [ VERIFICACIÓN DE AGRUPACIÓN ] ---")

total_rows = len(df)
nans_in_new_col = df['master_group_id'].isna().sum()
total_groups = df['master_group_id'].nunique()

# Contamos cuántas filas cayeron en cada regla (esto debe coincidir con tu auditoría anterior)
p_rows = df[df['master_group_id'].str.startswith('p_')].shape[0]
l_rows = df[df['master_group_id'].str.startswith('l_')].shape[0]
i_rows = df[df['master_group_id'].str.startswith('i_')].shape[0]

print(f"   - Filas totales procesadas: {total_rows:,}")
print(f"   - Nulos (NaN) en 'master_group_id': {nans_in_new_col} (Debe ser 0)")
print(f"   - Total de Grupos Únicos creados: {total_groups:,}")
print("   -------------------------------------")
print("   CONTEO DE FILAS POR TIPO DE ID:")
print(f"   - (Nivel 1) Usando 'patient_id': {p_rows:,} filas")
print(f"   - (Nivel 2) Usando 'lesion_id':  {l_rows:,} filas")
print(f"   - (Nivel 3) Usando 'isic_id':    {i_rows:,} filas")
print(f"   - Total verificado: {p_rows + l_rows + i_rows:,}")


# --- Guardado ---
try:
    df.to_csv(FILE_OUT, index=False)
    print(f"\n✅ ¡ÉXITO! Se ha guardado el nuevo archivo con IDs maestros en:")
    print(f"   -> '{FILE_OUT}'")
except Exception as e:
    print(f"\n❌ ERROR AL GUARDAR: {e}")

print("\n>>> 🎉 AGRUPACIÓN INTELIGENTE TERMINADA.")

>>> 🚀 INICIANDO AGRUPACIÓN INTELIGENTE (v3.0) en 'metadata_malignant.csv'...
✅ Archivo 'metadata_malignant.csv' cargado. 27,069 filas totales.
   -> Aplicando lógica de agrupación jerárquica...

--- [ VERIFICACIÓN DE AGRUPACIÓN ] ---
   - Filas totales procesadas: 27,069
   - Nulos (NaN) en 'master_group_id': 0 (Debe ser 0)
   - Total de Grupos Únicos creados: 11,983
   -------------------------------------
   CONTEO DE FILAS POR TIPO DE ID:
   - (Nivel 1) Usando 'patient_id': 6,619 filas
   - (Nivel 2) Usando 'lesion_id':  19,240 filas
   - (Nivel 3) Usando 'isic_id':    1,210 filas
   - Total verificado: 27,069

✅ ¡ÉXITO! Se ha guardado el nuevo archivo con IDs maestros en:
   -> 'metadata_malignant_grouped.csv'

>>> 🎉 AGRUPACIÓN INTELIGENTE TERMINADA.


## Estudio de patient_id y lesion_id despues de la fusion

In [8]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
# Este es el NUEVO archivo que generó el script v3.0
FILE_TO_CHECK = 'metadata_malignant_grouped.csv' 
COLS_TO_AUDIT = ['master_group_id', 'patient_id', 'lesion_id']

print(f">>> 🔬 AUDITORÍA DE VERIFICACIÓN de '{FILE_TO_CHECK}'...")

# ==============================================================================
# 2. CARGA Y ANÁLISIS
# ==============================================================================
try:
    # Cargamos solo las columnas clave para la auditoría
    df = pd.read_csv(FILE_TO_CHECK, usecols=COLS_TO_AUDIT, low_memory=False)
    total_rows = len(df)
    print(f"✅ Archivo cargado. Total de {total_rows:,} filas (imágenes) a analizar.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{FILE_TO_CHECK}'.")
    print("Asegúrate de haber ejecutado el script de 'agrupación inteligente' (v3.0) primero.")
    exit()
except ValueError:
    print(f"❌ ERROR: El archivo '{FILE_TO_CHECK}' no contiene las columnas {COLS_TO_AUDIT}.")
    exit()
except Exception as e:
    print(f"❌ ERROR inesperado: {e}")
    exit()

# --- 1. Verificación de Nulos en 'master_group_id' ---
print("\n--- [1] Verificación de 'master_group_id' (El nuevo ID inteligente) ---")
# Forzamos a string para que 'isna()' detecte cualquier problema
df['master_group_id'] = df['master_group_id'].astype(str)
nans_master_id = df['master_group_id'].isna().sum()
total_groups = df['master_group_id'].nunique()

print(f"   - Filas con 'master_group_id' Nulo (NaN): {nans_master_id}  (Debe ser 0)")
if nans_master_id == 0:
    print("   -> ✅ ¡ÉXITO! No hay IDs maestros nulos.")
else:
    print("   -> ❌ FALLO: Todavía hay IDs nulos.")

print(f"\n   - Total de Grupos Únicos creados: {total_groups:,}")
print("     (Este número debe ser mucho mayor que los 3,000+ pacientes originales)")


# --- 2. Verificación de la Lógica Jerárquica (Prefijos) ---
print("\n--- [2] Verificación de la Lógica de Creación de IDs ---")

p_rows = df[df['master_group_id'].str.startswith('p_')].shape[0]
l_rows = df[df['master_group_id'].str.startswith('l_')].shape[0]
i_rows = df[df['master_group_id'].str.startswith('i_')].shape[0]

print("   CONTEO DE FILAS POR TIPO DE ID:")
print(f"   - (Nivel 1: 'p_') Usando 'patient_id': {p_rows:,} filas")
print(f"   - (Nivel 2: 'l_') Usando 'lesion_id':  {l_rows:,} filas")
print(f"   - (Nivel 3: 'i_') Usando 'isic_id':    {i_rows:,} filas")

# Verificación cruzada con tu auditoría anterior (Total: 27,069)
# p_rows = Total - patient_id_nans = 27069 - 20450 = 6619
# l_rows = patient_id_nans - both_nans = 20450 - 1210 = 19240
# i_rows = both_nans = 1210
print("\n   VERIFICACIÓN LÓGICA (cifras esperadas según tu auditoría):")
print(f"     Esperábamos p_rows = 6,619.  Resultado: {'✅ OK' if p_rows == 6619 else '⚠️ REVISAR'}")
print(f"     Esperábamos l_rows = 19,240. Resultado: {'✅ OK' if l_rows == 19240 else '⚠️ REVISAR'}")
print(f"     Esperábamos i_rows = 1,210.  Resultado: {'✅ OK' if i_rows == 1210 else '⚠️ REVISAR'}")


# --- 3. Muestra de 'patient_id' y 'lesion_id' originales ---
print("\n--- [3] Muestra de IDs Originales (para referencia) ---")
print("\n   Muestra 'patient_id' (10 primeros únicos, sin nulos):")
# Convertimos a string y filtramos 'nan' para una muestra limpia
patient_id_sample = df['patient_id'].dropna().astype(str).unique()[:10]
print(f"   {patient_id_sample}")

print("\n   Muestra 'lesion_id' (10 primeros únicos, sin nulos):")
lesion_id_sample = df['lesion_id'].dropna().astype(str).unique()[:10]
print(f"   {lesion_id_sample}")


print("\n>>> 🎉 AUDITORÍA 'master_group_id' TERMINADA.")

>>> 🔬 AUDITORÍA DE VERIFICACIÓN de 'metadata_malignant_grouped.csv'...
✅ Archivo cargado. Total de 27,069 filas (imágenes) a analizar.

--- [1] Verificación de 'master_group_id' (El nuevo ID inteligente) ---
   - Filas con 'master_group_id' Nulo (NaN): 0  (Debe ser 0)
   -> ✅ ¡ÉXITO! No hay IDs maestros nulos.

   - Total de Grupos Únicos creados: 11,983
     (Este número debe ser mucho mayor que los 3,000+ pacientes originales)

--- [2] Verificación de la Lógica de Creación de IDs ---
   CONTEO DE FILAS POR TIPO DE ID:
   - (Nivel 1: 'p_') Usando 'patient_id': 6,619 filas
   - (Nivel 2: 'l_') Usando 'lesion_id':  19,240 filas
   - (Nivel 3: 'i_') Usando 'isic_id':    1,210 filas

   VERIFICACIÓN LÓGICA (cifras esperadas según tu auditoría):
     Esperábamos p_rows = 6,619.  Resultado: ✅ OK
     Esperábamos l_rows = 19,240. Resultado: ✅ OK
     Esperábamos i_rows = 1,210.  Resultado: ✅ OK

--- [3] Muestra de IDs Originales (para referencia) ---

   Muestra 'patient_id' (10 primeros úni

## COMPROBAMOS QUE EL CONTENIDO DE TODAS LAS BD ES CORRECTO Y TODO ESTA BIEN PARA PODER JUNTAR LAS BD

In [11]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
# Apuntamos al NUEVO archivo que creamos con la agrupación inteligente
FILE_TO_CHECK = 'metadata_malignant_grouped.csv' 
COLS_TO_AUDIT = ['master_group_id', 'patient_id', 'lesion_id', 'isic_id', 'diagnosis_3', 'image_type']

print(f">>> 🔬 AUDITORÍA DE VERIFICACIÓN (v3.0) de '{FILE_TO_CHECK}'...")

# ==============================================================================
# 2. CARGA Y ANÁLISIS
# ==============================================================================
try:
    df = pd.read_csv(FILE_TO_CHECK, usecols=COLS_TO_AUDIT, low_memory=False)
    total_rows = len(df)
    print(f"✅ Archivo cargado. Total de {total_rows:,} filas (imágenes) a analizar.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{FILE_TO_CHECK}'.")
    exit()
except ValueError:
    print(f"❌ ERROR: El archivo no contiene las columnas necesarias {COLS_TO_AUDIT}.")
    exit()
except Exception as e:
    print(f"❌ ERROR inesperado: {e}")
    exit()

# --- 1. Verificación de Nulos en 'master_group_id' ---
print("\n--- [1] Verificación de 'master_group_id' (El nuevo ID inteligente) ---")
nans_master_id = df['master_group_id'].isna().sum()
total_groups = df['master_group_id'].nunique()

print(f"   - Filas con 'master_group_id' Nulo (NaN): {nans_master_id}  (Debe ser 0)")
if nans_master_id == 0:
    print("   -> ✅ ¡ÉXITO! No hay IDs maestros nulos.")
else:
    print("   -> ❌ FALLO: Todavía hay IDs nulos.")
print(f"   - Total de Grupos Únicos creados: {total_groups:,}")


# --- 2. Verificación de la Lógica Jerárquica (Prefijos) ---
print("\n--- [2] Verificación de la Lógica de Creación de IDs ---")
p_rows = df[df['master_group_id'].str.startswith('p_')].shape[0]
l_rows = df[df['master_group_id'].str.startswith('l_')].shape[0]
i_rows = df[df['master_group_id'].str.startswith('i_')].shape[0]

print("   CONTEO DE FILAS POR TIPO DE ID:")
print(f"   - (Nivel 1: 'p_') Usando 'patient_id': {p_rows:,} filas")
print(f"   - (Nivel 2: 'l_') Usando 'lesion_id':  {l_rows:,} filas")
print(f"   - (Nivel 3: 'i_') Usando 'isic_id':    {i_rows:,} filas")
print(f"   - Total verificado: {p_rows + l_rows + i_rows:,} (Debe ser igual a {total_rows:,})")


# --- 3. Verificación de Contenido (Filtro y Clases) ---
print("\n--- [3] Verificación de Contenido para Fusión ---")
# A. Verificar filtro 'dermoscopic'
if 'image_type' in df.columns:
    count = (df['image_type'] == 'dermoscopic').sum()
    print(f"  ✅ OK: Se encontraron {count} imágenes 'dermoscopic' para usar.")
else:
    print("  ❌ ERROR CRÍTICO: Falta la columna 'image_type'.")

# B. Verificar diagnósticos clave en 'diagnosis_3'
if 'diagnosis_3' in df.columns:
    diag_vals = df['diagnosis_3'].dropna().astype(str).str.lower().unique()
    claves = ['melanoma', 'basal cell', 'squamous']
    found_all = True
    for k in claves:
        if not any(k in s for s in diag_vals):
            print(f"    ⚠️ ADVERTENCIA: No se detectó ninguna variante de '{k}'.")
            found_all = False
    if found_all:
        print("  ✅ OK: Todos los diagnósticos clave (MEL, BCC, SCC) están presentes.")
else:
     print("  ❌ ERROR CRÍTICO: Falta la columna 'diagnosis_3'.")


print("\n>>> 🎉 AUDITORÍA 'master_group_id' TERMINADA.")

>>> 🔬 AUDITORÍA DE VERIFICACIÓN (v3.0) de 'metadata_malignant_grouped.csv'...
✅ Archivo cargado. Total de 27,069 filas (imágenes) a analizar.

--- [1] Verificación de 'master_group_id' (El nuevo ID inteligente) ---
   - Filas con 'master_group_id' Nulo (NaN): 0  (Debe ser 0)
   -> ✅ ¡ÉXITO! No hay IDs maestros nulos.
   - Total de Grupos Únicos creados: 11,983

--- [2] Verificación de la Lógica de Creación de IDs ---
   CONTEO DE FILAS POR TIPO DE ID:
   - (Nivel 1: 'p_') Usando 'patient_id': 6,619 filas
   - (Nivel 2: 'l_') Usando 'lesion_id':  19,240 filas
   - (Nivel 3: 'i_') Usando 'isic_id':    1,210 filas
   - Total verificado: 27,069 (Debe ser igual a 27,069)

--- [3] Verificación de Contenido para Fusión ---
  ✅ OK: Se encontraron 21578 imágenes 'dermoscopic' para usar.
  ✅ OK: Todos los diagnósticos clave (MEL, BCC, SCC) están presentes.

>>> 🎉 AUDITORÍA 'master_group_id' TERMINADA.


## JUNTAMOS LAS BASES DE DATOS Y GENERAMOS DOS ARCHIVOS, UNO PARA EL PLAN A Y OTRO PARA EL PLAN B

In [12]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 1. CONFIGURACIÓN DE RUTAS (v4.0)
# ==============================================================================
PATHS = {
    'ISIC_RICO': 'ISIC_2024_Training_Input/metadataIsic.csv',
    'ISIC_GT': 'ISIC_2024_Training_GroundTruth.csv',
    'ISIC_SUPP': 'ISIC_2024_Training_Supplement.csv',
    # ¡USAMOS EL NUEVO ARCHIVO AGRUPADO!
    'MALIGNANT_DB': 'metadata_malignant_grouped.csv' 
}

print(">>> 🚀 INICIANDO PROCESO DE FUSIÓN FINAL (v4.0) con Agrupación Inteligente...")

# ==============================================================================
# 2. DEFINICIÓN DE COLUMNAS FINALES (PLAN A vs PLAN B)
# ==============================================================================
# Columna de Agrupación Maestra
MASTER_GROUP_COL = 'master_group_id' # ¡Usaremos esta en lugar de 'patient_id'!

# Gestión (Esenciales para DataLoaders)
COLS_MGMT = ['isic_id', MASTER_GROUP_COL, 'patient_id', 'lesion_id', 'source_db', 'image_type']
# Targets (Se generan abajo)
COLS_TARGETS = ['head_A_label', 'head_B_label']
# Inputs Clínicos Base (Comunes y fiables)
COLS_BASE_CLINICAL = ['age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm']
# Inputs Experimentales (Dudosos - Solo Plan B)
COLS_EXPERIMENTAL = ['family_hx_mm', 'personal_hx_mm', 'mel_ulcer', 'fitzpatrick_skin_type']

# ==============================================================================
# 3. FUNCIONES DE ETIQUETADO (LÓGICA CLÍNICA 4-Clases)
# ==============================================================================
def normalize(text):
    return str(text).lower().strip() if pd.notna(text) else ""

def get_label_B_isic(row):
    """Etiquetado 4-clases para ISIC (Benignos fusionados)"""
    is_malignant = row.get('malignant') == 1.0
    diag3 = normalize(row.get('iddx_3'))
    diag_full = normalize(row.get('iddx_full'))

    if not is_malignant:
        return 0  # Clase 0: BEN (Todos los Benignos)
    else:
        if 'melanoma' in diag3 or 'melanoma' in diag_full: return 1    # Clase 1: MEL
        elif 'basal' in diag3 or 'basal' in diag_full: return 2        # Clase 2: BCC
        elif 'squamous' in diag3 or 'actinic' in diag3 or 'bowen' in diag3: return 3 # Clase 3: SCC+
        else: return 1 # Fallback

def get_label_B_malignant(row):
    """Etiquetado 4-clases para la BD Maligna (Re-indexada)"""
    diag3 = normalize(row.get('diagnosis_3'))
    if 'melanoma' in diag3: return 1       # Clase 1: MEL
    # Se añade 'basal cell' para asegurar cobertura
    elif 'basal' in diag3 or 'basal cell' in diag3: return 2 # Clase 2: BCC
    elif 'squamous' in diag3 or 'actinic' in diag3 or 'keratoacanthoma' in diag3: return 3 # Clase 3: SCC+
    else: return -1 # Descartar casos raros

# ==============================================================================
# 4. CARGA Y PREPARACIÓN DE ISIC (La parte pesada)
# ==============================================================================
print("\n[1/4] 🏗️ Construyendo núcleo ISIC (~400k)...")
try:
    df_rico = pd.read_csv(PATHS['ISIC_RICO'], low_memory=False)
    df_gt = pd.read_csv(PATHS['ISIC_GT'])
    df_supp = pd.read_csv(PATHS['ISIC_SUPP'], usecols=['isic_id', 'lesion_id', 'iddx_3', 'iddx_full'])
except Exception as e:
    print(f"❌ ERROR: No se pudo cargar uno de los archivos de ISIC. {e}")
    exit()

print("   -> Fusionando Metadatos Ricos + GT + Suplemento...")
df_isic = pd.merge(df_rico, df_gt, on='isic_id', how='inner')
df_isic = pd.merge(df_isic, df_supp, on='isic_id', how='left', suffixes=('', '_supp'))

# --- Lógica de Agrupación Inteligente para ISIC ---
# (Asumimos que ISIC sí tiene patient_id, pero aplicamos la misma lógica por robustez)
if 'lesion_id_supp' in df_isic.columns:
    df_isic['lesion_id'] = df_isic['lesion_id'].fillna(df_isic['lesion_id_supp'])

df_isic[MASTER_GROUP_COL] = np.where(
    df_isic['patient_id'].notna(), 'p_' + df_isic['patient_id'].astype(str),
    np.where(
        df_isic['lesion_id'].notna(), 'l_' + df_isic['lesion_id'].astype(str),
        'i_' + df_isic['isic_id'].astype(str)
    )
)
print("   -> 'master_group_id' creado para ISIC.")

print("   -> Generando etiquetas duales (A/B) para ISIC...")
df_isic['head_A_label'] = df_isic['malignant'].astype(float)
df_isic['head_B_label'] = df_isic.apply(get_label_B_isic, axis=1)
df_isic['source_db'] = 'ISIC'

COLS_TBP_LV = [c for c in df_isic.columns if c.startswith('tbp_lv_')]
print(f"   -> ✅ Detectadas {len(COLS_TBP_LV)} columnas biométricas (tbp_lv).")

# ==============================================================================
# 5. CARGA Y PREPARACIÓN DE MALIGNANT DB
# ==============================================================================
print("\n[2/4] 🩺 Preparando refuerzo MALIGNO (ya agrupado)...")
try:
    # ¡Cargamos el archivo que ya tiene el 'master_group_id' inteligente!
    df_mal = pd.read_csv(PATHS['MALIGNANT_DB'], low_memory=False)
except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{PATHS['MALIGNANT_DB']}'.")
    print("   Asegúrate de ejecutar primero el script de 'agrupación inteligente' v3.0.")
    exit()
except Exception as e:
    print(f"❌ ERROR inesperado al cargar la BD Maligna Agrupada: {e}")
    exit()
    
# VERIFICAR que la columna maestra existe
if MASTER_GROUP_COL not in df_mal.columns:
    print(f"❌ ERROR CRÍTICO: El archivo '{PATHS['MALIGNANT_DB']}' no contiene la columna '{MASTER_GROUP_COL}'.")
    print("   Asegúrate de ejecutar el script de agrupación v3.0.")
    exit()

n_orig = len(df_mal)
df_mal = df_mal[df_mal['image_type'] == 'dermoscopic'].copy()
print(f"   -> Filtro aplicado: {n_orig} total -> {len(df_mal)} dermoscópicas válidas.")

print("   -> Generando etiquetas duales (A/B) para BD Maligna...")
df_mal['head_A_label'] = 1.0 
df_mal['head_B_label'] = df_mal.apply(get_label_B_malignant, axis=1)
df_mal['source_db'] = 'MALIGNANT'
print(f"   -> ✅ 'master_group_id' cargado desde el archivo.")

# ==============================================================================
# 6. FUSIÓN MAESTRA E IMPUTACIÓN
# ==============================================================================
print("\n[3/4] 🔄 FUSIONANDO y aplicando regla 94%/6%...")
df_master = pd.concat([df_isic, df_mal], ignore_index=True)

print("   -> 💉 Imputando valores faltantes (Estrategia TFG)...")
df_master[COLS_TBP_LV] = df_master[COLS_TBP_LV].fillna(-1)
df_master['age_approx'] = df_master['age_approx'].fillna(df_master['age_approx'].median())
df_master['sex'] = df_master['sex'].fillna('unknown')
df_master['anatom_site_general'] = df_master['anatom_site_general'].fillna('unknown')
df_master['clin_size_long_diam_mm'] = df_master['clin_size_long_diam_mm'].fillna(-1)

# Limpieza de etiquetas no válidas (-1) y IDs maestros nulos (por si acaso)
df_master = df_master[df_master['head_B_label'] != -1]
df_master = df_master[df_master[MASTER_GROUP_COL].notna()]

# ==============================================================================
# 7. GENERACIÓN DE ARCHIVOS FINALES (V4)
# ==============================================================================
print("\n[4/4] 💾 Guardando MASTER DATASETS V4...")

# Definir listas exactas de columnas para cada plan
FINAL_COLS_A = list(set(COLS_MGMT + COLS_TARGETS + COLS_BASE_CLINICAL + COLS_TBP_LV))
FINAL_COLS_B = list(set(FINAL_COLS_A + COLS_EXPERIMENTAL))

FINAL_COLS_A = [c for c in FINAL_COLS_A if c in df_master.columns]
FINAL_COLS_B = [c for c in FINAL_COLS_B if c in df_master.columns]

# --- Guardar PLAN A (Asegurado) ---
df_plan_a = df_master[FINAL_COLS_A].copy()
df_plan_a.to_csv('MASTER_MULTITASK_V4_PLAN_A.csv', index=False)
print(f"   ✅ GUARDADO: 'MASTER_MULTITASK_V4_PLAN_A.csv' ({len(df_plan_a)} filas, {len(FINAL_COLS_A)} cols)")

# --- Guardar PLAN B (Experimental) ---
for col in COLS_EXPERIMENTAL:
    if col in df_master.columns:
        fill_val = -1 if pd.api.types.is_numeric_dtype(df_master[col]) else 'missing'
        df_master[col] = df_master[col].fillna(fill_val)
        
df_plan_b = df_master[FINAL_COLS_B].copy()
df_plan_b.to_csv('MASTER_MULTITASK_V4_PLAN_B.csv', index=False)
print(f"   ✅ GUARDADO: 'MASTER_MULTITASK_V4_PLAN_B.csv' ({len(df_plan_b)} filas, {len(FINAL_COLS_B)} cols)")

print("\n📊 --- RESUMEN FINAL DE CLASES (HEAD B - 4 Clases) ---")
clases_map = {
    0.0: 'BEN (Benigno Total)', 
    1.0: 'MEL (Melanoma)', 
    2.0: 'BCC (Basocelular)', 
    3.0: 'SCC+ (Escamoso)'
} 

conteo = df_master['head_B_label'].value_counts().sort_index()
for k, v in clases_map.items():
    print(f"   [Clase {int(k)}] {v}: {conteo.get(k, 0):,} imágenes")

print("\n>>> 🎉 ¡PROCESO TERMINADO! Tus datos V4 están listos para el split.")

>>> 🚀 INICIANDO PROCESO DE FUSIÓN FINAL (v4.0) con Agrupación Inteligente...

[1/4] 🏗️ Construyendo núcleo ISIC (~400k)...
   -> Fusionando Metadatos Ricos + GT + Suplemento...
   -> 'master_group_id' creado para ISIC.
   -> Generando etiquetas duales (A/B) para ISIC...
   -> ✅ Detectadas 34 columnas biométricas (tbp_lv).

[2/4] 🩺 Preparando refuerzo MALIGNO (ya agrupado)...
   -> Filtro aplicado: 27069 total -> 21578 dermoscópicas válidas.
   -> Generando etiquetas duales (A/B) para BD Maligna...
   -> ✅ 'master_group_id' cargado desde el archivo.

[3/4] 🔄 FUSIONANDO y aplicando regla 94%/6%...
   -> 💉 Imputando valores faltantes (Estrategia TFG)...

[4/4] 💾 Guardando MASTER DATASETS V4...
   ✅ GUARDADO: 'MASTER_MULTITASK_V4_PLAN_A.csv' (422526 filas, 46 cols)
   ✅ GUARDADO: 'MASTER_MULTITASK_V4_PLAN_B.csv' (422526 filas, 50 cols)

📊 --- RESUMEN FINAL DE CLASES (HEAD B - 4 Clases) ---
   [Clase 0] BEN (Benigno Total): 400,666 imágenes
   [Clase 1] MEL (Melanoma): 10,628 imágenes
   [C

## COMPROBAMSO QUE LAS BD SE GENERARON BIEN

In [16]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 1. CONFIGURACIÓN DE ARCHIVOS A VERIFICAR
# ==============================================================================
# Apuntamos a los archivos V4 que acabas de generar
FILE_PLAN_A = 'MASTER_MULTITASK_V4_PLAN_A.csv'
FILE_PLAN_B = 'MASTER_MULTITASK_V4_PLAN_B.csv'
MASTER_GROUP_COL = 'master_group_id'

print(">>> 🔬 INICIANDO AUDITORÍA FINAL (v4.1) DE LOS ARCHIVOS MAESTROS V4...")

def audit_master_db(file_path, plan_name, is_plan_b=False):
    """Realiza una auditoría completa sobre el archivo CSV maestro final."""
    print(f"\n=======================================================")
    print(f"AUDITORÍA DE: {plan_name}")
    print(f"=======================================================")

    if not os.path.exists(file_path):
        print(f"❌ ERROR: No se encontró el archivo '{file_path}'.")
        return False

    try:
        df = pd.read_csv(file_path, low_memory=False)
        total_rows = len(df)
        print(f"✅ Archivo cargado correctamente ({total_rows:,} filas).")
    except Exception as e:
        print(f"❌ ERROR: No se pudo cargar el archivo. {e}")
        return False

    # --- 1. Verificación de Nulos en 'master_group_id' ---
    print(f"\n--- [1] Verificación del ID de Agrupación ('{MASTER_GROUP_COL}') ---")
    if MASTER_GROUP_COL not in df.columns:
        print(f"  -> ❌ ERROR CRÍTICO: ¡La columna '{MASTER_GROUP_COL}' no existe!")
        return False
        
    nans_master_id = df[MASTER_GROUP_COL].isna().sum()
    total_groups = df[MASTER_GROUP_COL].nunique()
    print(f"  - Filas con ID de grupo Nulo (NaN): {nans_master_id} (Debe ser 0)")
    print(f"  - Total de Grupos Únicos: {total_groups:,}")
    
    if nans_master_id > 0:
        print("  -> ❌ FALLO: Hay nulos en la columna de agrupación.")
        return False
    if total_groups < 10000: # Debería ser mucho mayor que los 3114 pacientes originales
        print(f"  -> ⚠️ ADVERTENCIA: El número de grupos ({total_groups}) es bajo. El 'mega-grupo' podría seguir existiendo.")
    else:
        print("  -> ✅ OK: El 'mega-grupo nan' ha sido disuelto exitosamente.")

    # --- 2. Verificación de la Lógica Jerárquica (Prefijos) ---
    print("\n--- [2] Verificación de la Lógica de Agrupación (Prefijos) ---")
    p_rows = df[df[MASTER_GROUP_COL].str.startswith('p_')].shape[0]
    l_rows = df[df[MASTER_GROUP_COL].str.startswith('l_')].shape[0]
    i_rows = df[df[MASTER_GROUP_COL].str.startswith('i_')].shape[0]
    print(f"  - Grupos 'p_' (patient_id): {p_rows:,} filas")
    print(f"  - Grupos 'l_' (lesion_id):  {l_rows:,} filas")
    print(f"  - Grupos 'i_' (isic_id):    {i_rows:,} filas")
    
    if (p_rows + l_rows + i_rows) != total_rows:
        print("  -> ❌ FALLO: El conteo de prefijos no coincide con el total de filas.")
        return False
    else:
        print("  -> ✅ OK: La lógica de agrupación jerárquica es correcta.")

    # --- 3. Verificación de Etiquetas (Head B - 4 Clases) ---
    print("\n--- [3] Verificación de Distribución de Clases (Head B) ---")
    clases_map = {0.0: 'BEN', 1.0: 'MEL', 2.0: 'BCC', 3.0: 'SCC+'}
    conteo_abs = df['head_B_label'].value_counts()
    print(conteo_abs.rename(index=clases_map).sort_index())
    
    if len(conteo_abs) != 4:
        print(f"  -> ❌ FALLO: Se esperaban 4 clases, pero se encontraron {len(conteo_abs)}.")
        return False
    else:
        print("  -> ✅ OK: La estructura de 4 clases es correcta.")

    # --- 4. Verificación de Imputación TBP-LV (Regla 94%/6%) ---
    print("\n--- [4] Verificación de Imputación de Metadatos (-1) ---")
    tbp_col = 'tbp_lv_areaMM2' # Columna de muestra
    if tbp_col in df.columns:
        nans_tbp = df[tbp_col].isna().sum()
        imputed_count = (df[tbp_col] == -1.0).sum()
        print(f"  - Valores NaN remanentes en TBP-LV: {nans_tbp} (Debe ser 0)")
        print(f"  - Filas imputadas con -1: {imputed_count:,} (Debe ser ~21k)")
        
        if nans_tbp > 0 or imputed_count < 20000:
            print("  -> ❌ FALLO: La imputación de -1 parece incorrecta.")
            return False
        else:
            print("  -> ✅ OK: La imputación estratégica (94/6) es correcta.")
    else:
        print(f"  -> ❌ FALLO: No se encontró la columna de verificación '{tbp_col}'.")
        return False
        
    return True # Si todo pasó

# Ejecutar la auditoría
print("Realizando auditoría del Plan A...")
A_ok = audit_master_db(FILE_PLAN_A, "MASTER_MULTITASK_V4_PLAN_A.csv", is_plan_b=False)

print("\n\nRealizando auditoría del Plan B...")
B_ok = audit_master_db(FILE_PLAN_B, "MASTER_MULTITASK_V4_PLAN_B.csv", is_plan_b=True)

print("\n" + "="*50)
print("VEREDICTO FINAL DE LA AUDITORÍA V4.1:")
if A_ok and B_ok:
    print("✅ ¡ÉXITO! Ambos archivos maestros (Plan A y Plan B) están 100% correctos y listos para el SPLIT.")
else:
    print("❌ ERROR: Uno o ambos archivos maestros fallaron la auditoría. Revisa los logs de arriba.")
print("="*50)

>>> 🔬 INICIANDO AUDITORÍA FINAL (v4.1) DE LOS ARCHIVOS MAESTROS V4...
Realizando auditoría del Plan A...

AUDITORÍA DE: MASTER_MULTITASK_V4_PLAN_A.csv
✅ Archivo cargado correctamente (422,526 filas).

--- [1] Verificación del ID de Agrupación ('master_group_id') ---
  - Filas con ID de grupo Nulo (NaN): 0 (Debe ser 0)
  - Total de Grupos Únicos: 11,895
  -> ✅ OK: El 'mega-grupo nan' ha sido disuelto exitosamente.

--- [2] Verificación de la Lógica de Agrupación (Prefijos) ---
  - Grupos 'p_' (patient_id): 405,852 filas
  - Grupos 'l_' (lesion_id):  15,592 filas
  - Grupos 'i_' (isic_id):    1,082 filas
  -> ✅ OK: La lógica de agrupación jerárquica es correcta.

--- [3] Verificación de Distribución de Clases (Head B) ---
head_B_label
BCC       8759
BEN     400666
MEL      10628
SCC+      2473
Name: count, dtype: int64
  -> ✅ OK: La estructura de 4 clases es correcta.

--- [4] Verificación de Imputación de Metadatos (-1) ---
  - Valores NaN remanentes en TBP-LV: 0 (Debe ser 0)
  - Filas 

In [17]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 0. CONFIGURACIÓN
# ==============================================================================
# Usamos el Plan A (V4) porque contiene todas las filas
MASTER_DB_FILE = 'MASTER_MULTITASK_V4_PLAN_A.csv' 
# --- ¡CAMBIO IMPORTANTE! ---
# Ahora auditamos la nueva columna de agrupación inteligente
GROUP_COLUMN = 'master_group_id'

print(f">>> 📈 ANALIZANDO LA DISTRIBUCIÓN DE GRUPOS ('{GROUP_COLUMN}') en {MASTER_DB_FILE}...")
print("Cargando archivo (esto puede tardar un momento)...")

# ==============================================================================
# 1. CARGA Y CONTEO
# ==============================================================================
try:
    # Cargar solo la columna 'master_group_id' para ahorrar memoria
    df_groups = pd.read_csv(MASTER_DB_FILE, usecols=[GROUP_COLUMN], low_memory=False)
    print(f"✅ Archivo cargado. Total de {len(df_groups):,} imágenes (filas) a analizar.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{MASTER_DB_FILE}'.")
    exit()
except ValueError:
     print(f"❌ ERROR: El archivo '{MASTER_DB_FILE}' no contiene la columna '{GROUP_COLUMN}'.")
     exit()
except Exception as e:
    print(f"❌ ERROR inesperado: {e}")
    exit()

# Limpieza (aunque el script v4.0 ya no debería dejar nulos, esto es por seguridad)
df_groups[GROUP_COLUMN] = df_groups[GROUP_COLUMN].astype(str).fillna('MISSING_PID')

# Realizar el conteo de imágenes por grupo
print("\n--- [1] Realizando conteo (value_counts)... ---")
group_counts = df_groups[GROUP_COLUMN].value_counts()

# ==============================================================================
# 2. ANÁLISIS Y RESULTADOS
# ==============================================================================
print("\n--- [2] Resumen Estadístico (Imágenes por Grupo Maestro) ---")
print(f"   - Total Grupos Únicos: {len(group_counts):,}")
print(f"   - Total Imágenes Analizadas: {group_counts.sum():,}")
print(f"   - Media de imágenes por grupo: {group_counts.mean():.2f}")
print(f"   - Mediana de imágenes por grupo: {group_counts.median():.0f}")
print(f"   - Max (Grupo con más imágenes): {group_counts.max():,}")
print(f"   - Min (Grupo con menos imágenes): {group_counts.min():,}")

# Mostrar los "mega-grupos"
print("\n--- [3] TOP 15 Grupos con MÁS imágenes (Mega-Grupos) ---")
print(group_counts.head(15).to_string())

# Mostrar grupos con pocas imágenes
print("\n--- [4] Muestra de Grupos con MENOS imágenes (ej. huérfanos 'i_') ---")
print(group_counts.tail(15).to_string())

# ==============================================================================
# 3. GUARDADO DE "TODOS" LOS DATOS
# ==============================================================================
output_filename = 'master_group_image_counts.csv'
try:
    group_counts.to_csv(output_filename)
    print(f"\n--- [5] LISTA COMPLETA GUARDADA ---")
    print(f"   -> Se han guardado los conteos de TODOS los grupos en el archivo: '{output_filename}'")
except Exception as e:
    print(f"\n--- [5] ERROR AL GUARDAR ---")
    print(f"   -> No se pudo guardar el archivo CSV: {e}")

print("\n>>> 🎉 ANÁLISIS DE 'master_group_id' TERMINADO.")

>>> 📈 ANALIZANDO LA DISTRIBUCIÓN DE GRUPOS ('master_group_id') en MASTER_MULTITASK_V4_PLAN_A.csv...
Cargando archivo (esto puede tardar un momento)...
✅ Archivo cargado. Total de 422,526 imágenes (filas) a analizar.

--- [1] Realizando conteo (value_counts)... ---

--- [2] Resumen Estadístico (Imágenes por Grupo Maestro) ---
   - Total Grupos Únicos: 11,895
   - Total Imágenes Analizadas: 422,526
   - Media de imágenes por grupo: 35.52
   - Mediana de imágenes por grupo: 1
   - Max (Grupo con más imágenes): 9,184
   - Min (Grupo con menos imágenes): 1

--- [3] TOP 15 Grupos con MÁS imágenes (Mega-Grupos) ---
master_group_id
p_IP_1117889    9184
p_IP_5714646    6267
p_IP_3921915    5568
p_IP_7797815    4454
p_IP_9577633    3583
p_IP_5539318    2859
p_IP_9853536    2327
p_IP_5143034    2283
p_IP_0321326    2245
p_IP_5426188    2193
p_IP_1127121    2183
p_IP_1116526    2121
p_IP_2331257    1909
p_IP_6724798    1859
p_IP_3785831    1853

--- [4] Muestra de Grupos con MENOS imágenes (ej. hu

## CREAMOS EL SPLIT ENTRE TEST Y TRAIN (15% - 85%) & EL SPLIT ENTRE EL TRAINING SET EN 17%

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings
import os

# Ignorar advertencias
warnings.filterwarnings('ignore', category=UserWarning)

# ==============================================================================
# 0. CONFIGURACIÓN INICIAL (v1.3 - Apuntando a V4 y master_group_id)
# ==============================================================================
print(">>> 🚀 INICIANDO PIPELINE DE VERIFICACIÓN (v1.3)...")

MASTER_DB_FILE = 'MASTER_MULTITASK_V4_PLAN_A.csv' # <-- CAMBIO: Apunta a V4
TARGET_COLUMN = 'head_B_label' 
GROUP_COLUMN = 'master_group_id' # <-- CAMBIO: Usamos el ID inteligente
RANDOM_SEED = 42               

# Mapa de clases para hacer legibles los reportes
clases_map = {
    0.0: '0: BEN (Benigno)', 
    1.0: '1: MEL (Melanoma)', 
    2.0: '2: BCC (Basocelular)', 
    3.0: '3: SCC+ (Escamoso)'
}

try:
    df = pd.read_csv(MASTER_DB_FILE, low_memory=False)
    print(f"✅ Archivo '{MASTER_DB_FILE}' cargado. {len(df)} filas totales.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo '{MASTER_DB_FILE}'.")
    exit()
except Exception as e:
    print(f"❌ ERROR inesperado al cargar el archivo: {e}")
    exit()

# Identificar columnas automáticamente
# <-- CAMBIO: Añadido MASTER_GROUP_COL a la reserva
COLS_RESERVED = [TARGET_COLUMN, 'head_A_label', 'patient_id', 'master_group_id', 'isic_id', 'lesion_id', 'source_db', 'image_type']
COLS_NUMERIC = [col for col in df.columns if col not in COLS_RESERVED and pd.api.types.is_numeric_dtype(df[col])]
COLS_CATEGORICAL = [col for col in df.columns if col not in COLS_RESERVED and not pd.api.types.is_numeric_dtype(df[col])]

print(f"   -> {len(COLS_NUMERIC)} features numéricas detectadas (age, tbp_lv...)")
print(f"   -> {len(COLS_CATEGORICAL)} features categóricas detectadas (sex, anatom_site...)")

# Definir X (features), y (target), groups (pacientes)
X = df.drop(columns=[TARGET_COLUMN, 'head_A_label']) 
y = df[TARGET_COLUMN]

# ==============================================================================
#  (!!!!) CORRECCIÓN CRÍTICA Y VERIFICACIÓN (!!!!)
# ==============================================================================
# El script V4 ya debería haber limpiado esto, pero lo aseguramos.
groups = df[GROUP_COLUMN].astype(str).fillna('MISSING_PID')
print(f"   -> Columna '{GROUP_COLUMN}' (groups) verificada.")

# --- VERIFICACIÓN TAREA 4: MUESTRA DE 'master_group_id' ---
print("\n--- [VERIFICACIÓN TAREA 4: Muestra de 'master_group_id' limpios] ---")
print(np.unique(groups)[:10]) # Muestra los primeros 10 IDs únicos
print("   -> (Deberías ver IDs con prefijos 'i_...', 'l_...' y 'p_...')")


# ==============================================================================
# FASE 1: PARTICIÓN MAESTRA (80% Desarrollo / 20% Test)
# ==============================================================================
print("\n--- [FASE 1: PARTICIÓN MAESTRA (80/20)] ---")
master_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

try:
    # <-- CAMBIO CRÍTICO: Índices asignados correctamente (train=dev, test=test)
    dev_indices, test_indices = next(master_splitter.split(X, y, groups))
except ValueError as e:
    print(f"\n❌ ERROR CRÍTICO EN EL SPLIT: {e}")
    print("   Esto puede pasar si una clase (ej. SCC+) tiene menos de 5 grupos únicos (n_splits).")
    exit()

dev_data = df.iloc[dev_indices].copy()
test_data = df.iloc[test_indices].copy()

print(f"✅ División Maestra Creada:")
print(f"   -> {len(dev_data)} filas para Desarrollo (80%)")
print(f"   -> {len(test_data)} filas para Prueba (20%)")

# --- VERIFICACIÓN TAREA 1: DISTRIBUCIÓN DE CLASES (80/20) ---
print("\n--- [VERIFICACIÓN TAREA 1: Distribución de Clases (Maestra)] ---")
print("  Distribución en DEVELOPMENT SET (80%):")
print((dev_data[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100).rename(clases_map))
print("\n  Distribución en TEST SET (20%):")
print((test_data[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100).rename(clases_map))
print("   -> (Ahora ambas distribuciones deberían ser CASI IDÉNTICAS)")

# Guardamos los sets
dev_data.to_csv('development_data.csv', index=False)
test_data.to_csv('test_data.csv', index=False)
print("\n   -> 'development_data.csv' y 'test_data.csv' guardados.")


# ==============================================================================
# FASE 2: BUCLE K-FOLD SOBRE EL SET DE DESARROLLO (80%)
# ==============================================================================
print("\n--- [FASE 2: BUCLE K-FOLD (sobre el 80% de Desarrollo)] ---")

N_SPLITS_CV = 5
kfold_splitter = StratifiedGroupKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_SEED)

X_dev = dev_data.drop(columns=[TARGET_COLUMN, 'head_A_label'])
y_dev = dev_data[TARGET_COLUMN]
groups_dev = dev_data[GROUP_COLUMN].astype(str).fillna('MISSING_PID')

# --- VERIFICACIÓN TAREA 3: ÍNDICES DE COLUMNAS A CHEQUEAR ---
# Guardamos los índices de las columnas que queremos monitorizar
age_idx = COLS_NUMERIC.index('age_approx')
area_idx = COLS_NUMERIC.index('tbp_lv_areaMM2')
imputed_idx = COLS_NUMERIC.index('tbp_lv_A') # Una de las 34 imputadas

# Iniciar el bucle de Validación Cruzada
for fold, (train_idx, val_idx) in enumerate(kfold_splitter.split(X_dev, y_dev, groups_dev)):
    print(f"\n   --- FOLD {fold+1}/{N_SPLITS_CV} ---")
    
    # 1. Dividir este fold
    X_train_fold, y_train_fold = X_dev.iloc[train_idx], y_dev.iloc[train_idx]
    X_val_fold, y_val_fold = X_dev.iloc[val_idx], y_dev.iloc[val_idx]
    
    print(f"     -> Train: {len(X_train_fold)} muestras | Val: {len(X_val_fold)} muestras")

    # --- VERIFICACIÓN TAREA 2: DISTRIBUCIÓN DE CLASES (K-FOLD) ---
    print("\n     --- [VERIFICACIÓN TAREA 2: Distribución de Clases (Fold)] ---")
    print("       Train Fold %:")
    print((y_train_fold.value_counts(normalize=True).sort_index() * 100).rename(clases_map))
    print("\n       Val Fold %:")
    print((y_val_fold.value_counts(normalize=True).sort_index() * 100).rename(clases_map))
    print("       -> (Ambas distribuciones deben ser casi idénticas)")

    # ==========================================================================
    # FASE 3: NORMALIZACIÓN (DENTRO DEL BUCLE)
    # ==========================================================================
    print("\n     -> [FASE 3] Aplicando Normalización (Z-score y OHE)...")
    
    num_scaler = StandardScaler()
    cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    num_scaler.fit(X_train_fold[COLS_NUMERIC])
    cat_encoder.fit(X_train_fold[COLS_CATEGORICAL])

    X_train_num_norm = num_scaler.transform(X_train_fold[COLS_NUMERIC])
    X_val_num_norm = num_scaler.transform(X_val_fold[COLS_NUMERIC])
    X_train_cat_norm = cat_encoder.transform(X_train_fold[COLS_CATEGORICAL])
    X_val_cat_norm = cat_encoder.transform(X_val_fold[COLS_CATEGORICAL])

    X_train_processed = np.hstack((X_train_num_norm, X_train_cat_norm))
    X_val_processed = np.hstack((X_val_num_norm, X_val_cat_norm))
    
    # --- VERIFICACIÓN TAREA 3: CHEQUEO DE NORMALIZACIÓN (Z-SCORE) ---
    print("\n     --- [VERIFICACIÓN TAREA 3: Normalización Z-Score] ---")
    print(f"       'age_approx' (Train):   Mean={X_train_num_norm[:, age_idx].mean():.2f}, Std={X_train_num_norm[:, age_idx].std():.2f} (Debe ser 0.00, 1.00)")
    print(f"       'age_approx' (Val):     Mean={X_val_num_norm[:, age_idx].mean():.2f}, Std={X_val_num_norm[:, age_idx].std():.2f} (Debe ser cercano a 0 y 1)")
    print(f"       'tbp_lv_areaMM2' (Train): Mean={X_train_num_norm[:, area_idx].mean():.2f}, Std={X_train_num_norm[:, area_idx].std():.2f} (Debe ser 0.00, 1.00)")
    print(f"       'tbp_lv_areaMM2' (Val):   Mean={X_val_num_norm[:, area_idx].mean():.2f}, Std={X_val_num_norm[:, area_idx].std():.2f} (Debe ser cercano a 0 y 1)")
    print(f"       'tbp_lv_A' (Imputada -1): Mean={X_train_num_norm[:, imputed_idx].mean():.2f}, Std={X_train_num_norm[:, imputed_idx].std():.2f} (Debe ser 0.00, 1.00)")


    print(f"\n     -> Datos listos para el MLP:")
    print(f"        Train shape final: {X_train_processed.shape}")
    print(f"        Val shape final:   {X_val_processed.shape}")

    # ==========================================================================
    # FASE 4: ENTRENAMIENTO (Placeholder)
    # ==========================================================================
    print("     -> [FASE 4] TODO: Entrenar modelo (ARP-ViT-CNN + MLP) aquí...")
    print("     -> ...")


print("\n>>> ✅ Bucle de Validación Cruzada completado.")

# ==============================================================================
# FASE 5: NORMALIZACIÓN DEL TEST SET (Para el Examen Final)
# ==============================================================================
print("\n--- [FASE 5: Preparación del TEST SET (Examen Final)] ---")
print("   (Esto se hace solo UNA VEZ al final del TFG)")

# 1. APRENDER (fit) - ¡Usando TODOS los datos de DESARROLLO (el 80%)!
final_num_scaler = StandardScaler().fit(X_dev[COLS_NUMERIC])
final_cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(X_dev[COLS_CATEGORICAL])

# 2. TRANSFORMAR (transform) - Aplicar al TEST SET
X_test = test_data.drop(columns=[TARGET_COLUMN, 'head_A_label'])
X_test_num_norm = final_num_scaler.transform(X_test[COLS_NUMERIC])
X_test_cat_norm = final_cat_encoder.transform(X_test[COLS_CATEGORICAL])

X_test_processed = np.hstack((X_test_num_norm, X_test_cat_norm))

# --- VERIFICACIÓN TAREA 3 (FINAL): CHEQUEO TEST SET ---
print("\n   --- [VERIFICACIÓN TAREA 3: Normalización Test Set] ---")
print(f"     'age_approx' (Test):   Mean={X_test_num_norm[:, age_idx].mean():.2f}, Std={X_test_num_norm[:, age_idx].std():.2f}")
print(f"     'tbp_lv_areaMM2' (Test): Mean={X_test_num_norm[:, area_idx].mean():.2f}, Std={X_test_num_norm[:, area_idx].std():.2f}")
print(f"     -> (La media/std del Test Set NO serán 0.00/1.00, y eso es CORRECTO)")
print(f"     -> Test set final listo para evaluación: {X_test_processed.shape}")

print("\n>>> 🎉 PIPELINE DE PREPROCESAMIENTO COMPLETO Y CORRECTO.")

>>> 🚀 INICIANDO PIPELINE DE VERIFICACIÓN (v1.3)...
✅ Archivo 'MASTER_MULTITASK_V4_PLAN_A.csv' cargado. 422526 filas totales.
   -> 34 features numéricas detectadas (age, tbp_lv...)
   -> 4 features categóricas detectadas (sex, anatom_site...)
   -> Columna 'master_group_id' (groups) verificada.

--- [VERIFICACIÓN TAREA 4: Muestra de 'master_group_id' limpios] ---
['i_ISIC_0000002' 'i_ISIC_0000004' 'i_ISIC_0000013' 'i_ISIC_0000022'
 'i_ISIC_0000026' 'i_ISIC_0000029' 'i_ISIC_0000030' 'i_ISIC_0000031'
 'i_ISIC_0000035' 'i_ISIC_0000036']
   -> (Deberías ver IDs con prefijos 'i_...', 'l_...' y 'p_...')

--- [FASE 1: PARTICIÓN MAESTRA (80/20)] ---
✅ División Maestra Creada:
   -> 331333 filas para Desarrollo (80%)
   -> 91193 filas para Prueba (20%)

--- [VERIFICACIÓN TAREA 1: Distribución de Clases (Maestra)] ---
  Distribución en DEVELOPMENT SET (80%):
head_B_label
0: BEN (Benigno)        94.735508
1: MEL (Melanoma)        2.537326
2: BCC (Basocelular)     2.123543
3: SCC+ (Escamoso)      